In [0]:
from pyspark.sql.functions import col, concat, lit, to_timestamp, date_format, when, array, array_sort, concat_ws, broadcast

catalog_name = "26100355-pa2"
bronze_table = f"`{catalog_name}`.`bronze`.`flights`"
silver_table = f"`{catalog_name}`.`silver`.`flights`"
checkpoint_path = f"/Volumes/{catalog_name}/bronze/temp/checkpoints/silver_flights"

print("Silver Layer processing")

bronze_stream = spark.readStream.table(bronze_table)

silver_part1_df = (bronze_stream
    .withColumn("delay", col("delay").cast("integer"))
    .withColumn("distance", col("distance").cast("integer"))
    .withColumn("origin", col("origin").cast("string"))
    .withColumn("destination", col("destination").cast("string"))
    .withColumn("event_timestamp", to_timestamp(concat(lit("2025"), col("date")), "yyyyMMddHHmm"))
    .withColumn("date", date_format(col("event_timestamp"), "yyyy-MM-dd HH:mm:ss"))
)

ref_df = spark.read.csv("/databricks-datasets/flights/airport-codes-na.txt", header=True, sep="\t")

org_ref = broadcast(ref_df).alias("org")
dst_ref = broadcast(ref_df).alias("dst")

silver_part2_df = (silver_part1_df
    .join(org_ref, col("origin") == col("org.IATA"), "left")
    .join(dst_ref, col("destination") == col("dst.IATA"), "left")
    .withColumn("departure", concat_ws(", ", col("org.City"), col("org.State")))
    .withColumn("arrival", concat_ws(", ", col("dst.City"), col("dst.State")))
    .filter((col("departure") != "") & (col("arrival") != ""))
    .withColumn("take_off_status", 
        when(col("delay") < 0, "Early")
        .when((col("delay") >= 0) & (col("delay") < 10), "On Time")
        .when((col("delay") >= 10) & (col("delay") < 30), "Late")
        .when(col("delay") >= 30, "Delay")
    )
    .withColumn("standardized_route", concat_ws("-", array_sort(array(col("origin"), col("destination")))))
    .withColumn("seq", lit(3))
    .select(
        "date", "departure", "arrival", "standardized_route", 
        "distance", "delay", "take_off_status", 
        "ingestion_timestamp", "event_timestamp", "seq"
    )
)

write_query = (silver_part2_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

write_query.awaitTermination()
print("Silver Layer complete.")

Silver Layer processing
Silver Layer complete.



A broadcast join optimizes performance by sending a complete copy of a small table to every worker node in the cluster. This allows the join to happen locally on each machine, completely eliminating the need to "shuffle" the large table's data across the network, which is typically a very slow and expensive operation.

In [0]:
catalog_name = "26100355-pa2"
#work around by making chek point to display stream data
display_checkpoint_1 = f"/Volumes/{catalog_name}/bronze/temp/checkpoints/display_silver_1"
display_checkpoint_2 = f"/Volumes/{catalog_name}/bronze/temp/checkpoints/display_silver_2"
display(silver_part1_df, checkpointLocation=display_checkpoint_1)
display(silver_part2_df, checkpointLocation=display_checkpoint_2)
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`silver`.`flights` LIMIT 10"))

Checkpointing to /Volumes/26100355-pa2/bronze/temp/checkpoints/display_silver_1


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4574403004014929>, line 5
      3 display_checkpoint_1 = f"/Volumes/{catalog_name}/bronze/temp/checkpoints/display_silver_1"
      4 display_checkpoint_2 = f"/Volumes/{catalog_name}/bronze/temp/checkpoints/display_silver_2"
----> 5 display(silver_part1_df, checkpointLocation=display_checkpoint_1)
      6 display(silver_part2_df, checkpointLocation=display_checkpoint_2)
      7 display(spark.sql(f"SELECT * FROM `{catalog_name}`.`silver`.`flights` LIMIT 10"))

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/pytho

# **GOLD LAYER**

In [0]:
from pyspark.sql.functions import col, window, avg, sum, expr, round, date_format, count

catalog_name = "26100355-pa2"
silver_table = f"`{catalog_name}`.`silver`.`flights`"
base_checkpoint = f"/Volumes/{catalog_name}/bronze/temp/checkpoints"
silver_stream = spark.readStream.table(silver_table).withWatermark("event_timestamp", "7 days")

req1_df = (silver_stream
    .groupBy(window(col("event_timestamp"), "30 days"))
    .agg(round(avg(col("delay") / 60), 2).alias("moving_avg_delay"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "moving_avg_delay"
    )
)
req1_query = (req1_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/req1_monthly_delays")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`monthly_delays`")
)

req2_df = (silver_stream
    .groupBy(window(col("event_timestamp"), "14 days"))
    .agg(round(sum(col("distance")), 0).alias("total_distance"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "total_distance"
    )
)
req2_query = (req2_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/req2_biweekly_distance")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`biweekly_distance`")
)

req3_df = (silver_stream
    .withColumn("delay_hours", col("delay") / 60)
    .groupBy(window(col("event_timestamp"), "30 days"))
    .agg(
        expr("max_by(standardized_route, delay_hours)").alias("worst_route"),
        round(expr("max(delay_hours)"), 2).alias("max_delay")
    )
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "worst_route",
        "max_delay"
    )
)
req3_query = (req3_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/req3_monthly_max_delay")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`monthly_max_delay`")
)

task4_df = (silver_stream
    .groupBy(window(col("event_timestamp"), "1 hour"))
    .agg(count("*").alias("flight_count"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "flight_count"
    )
)
task4_query = (task4_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/task4_hourly_counts")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`hourly_flight_counts`")
)

#run
req1_query.awaitTermination()
req2_query.awaitTermination()
req3_query.awaitTermination()
task4_query.awaitTermination()


In [0]:
print("Requirement 1")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`monthly_delays` ORDER BY window_start LIMIT 5"))

print("Requirement 2")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`biweekly_distance` ORDER BY window_start LIMIT 5"))

print("Requirement 3")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`monthly_max_delay` ORDER BY window_start LIMIT 5"))

print("Task 4:")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`hourly_flight_counts` ORDER BY window_start LIMIT 5"))

Requirement 1


window_start,window_end,moving_avg_delay
2024-12-13 00:00:00,2025-01-12 00:00:00,0.44
2025-01-12 00:00:00,2025-02-11 00:00:00,0.17
2025-02-11 00:00:00,2025-03-13 00:00:00,0.18
2025-03-13 00:00:00,2025-04-12 00:00:00,0.15


Requirement 2


window_start,window_end,total_distance
2024-12-19 00:00:00,2025-01-02 00:00:00,9967392
2025-01-02 00:00:00,2025-01-16 00:00:00,143764431
2025-01-16 00:00:00,2025-01-30 00:00:00,137124712
2025-01-30 00:00:00,2025-02-13 00:00:00,137128874
2025-02-13 00:00:00,2025-02-27 00:00:00,145578616


Requirement 3


window_start,window_end,worst_route,max_delay
2024-12-13 00:00:00,2025-01-12 00:00:00,CMH-LAX,24.35
2025-01-12 00:00:00,2025-02-11 00:00:00,DFW-FLL,27.27
2025-02-11 00:00:00,2025-03-13 00:00:00,DFW-TPA,27.37
2025-03-13 00:00:00,2025-04-12 00:00:00,DFW-ELP,24.5


Task 4:


window_start,window_end,flight_count
2025-01-01 00:00:00,2025-01-01 01:00:00,20
2025-01-01 01:00:00,2025-01-01 02:00:00,8
2025-01-01 02:00:00,2025-01-01 03:00:00,3
2025-01-01 05:00:00,2025-01-01 06:00:00,136
2025-01-01 06:00:00,2025-01-01 07:00:00,634
